# Session 1.2 -- Batch Processing and Extraction

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Define a Pydantic schema** that validates structured data extracted from unstructured text
2. **Build a single-document extraction function** that sends unstructured text to Claude and returns validated, structured output
3. **Process a batch of documents** through the extraction function with error handling (failures logged, not crashed)

## What You Are Working With

- `src/s1_extraction/schemas.py` -- Pydantic models for structured extraction (provided complete)
- `src/s1_extraction/extractor.py` -- Extraction functions: `extract_single()` and `extract_batch()` (provided complete)
- `data/northbrook/` -- Corpus of memos, meeting notes, financial reports, and policies from Northbrook Partners

You **import and use** the pre-built modules. You do not need to build them from scratch.

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import json
from pathlib import Path
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

# Pre-built extraction modules
from src.s1_extraction.schemas import DocumentMeta, MemoExtraction, MeetingExtraction, PolicyExtraction
from src.s1_extraction.extractor import extract_single, extract_batch, ExtractionResult

print("All imports loaded successfully.")

---

## 2. Schema Exploration

Pydantic schemas define the **contract** between Claude's output and our code. Each field has a type, a name, and a description. The descriptions are not just documentation -- they become part of the prompt that Claude reads during extraction.

Let's look at what we are working with.

In [ ]:
# View all fields on MemoExtraction -- names, types, and whether they are required
for name, field_info in MemoExtraction.model_fields.items():
    required = "REQUIRED" if field_info.is_required() else f"optional (default: {field_info.default})"
    print(f"  {name:20s}  type={field_info.annotation}  [{required}]")
    print(f"  {'':20s}  description: {field_info.description}")
    print()

In [ ]:
# This is the JSON Schema that gets sent to Claude as part of the prompt.
# Notice how each field's description appears in the schema.
schema = MemoExtraction.model_json_schema()
print(json.dumps(schema, indent=2))

**Key insight:** The `description` on each field does double duty. It documents the schema for Python developers AND it tells Claude what to extract. Better descriptions lead to better extraction.

In [ ]:
# Compare schemas: MeetingExtraction has different fields for a different document type
print("=== MeetingExtraction fields ===")
for name, field_info in MeetingExtraction.model_fields.items():
    required = "REQUIRED" if field_info.is_required() else "optional"
    print(f"  {name:20s}  [{required}]  -- {field_info.description}")

print()
print("=== PolicyExtraction fields ===")
for name, field_info in PolicyExtraction.model_fields.items():
    required = "REQUIRED" if field_info.is_required() else "optional"
    print(f"  {name:20s}  [{required}]  -- {field_info.description}")

In [ ]:
# Create a valid MemoExtraction by hand to see what a correctly shaped object looks like
valid_memo = MemoExtraction(
    author="Sarah Chen",
    date="2025-03-15",
    subject="Q1 Budget Review",
    key_points=["Approved marketing spend increase of 12%", "Delayed office renovation to Q3"],
    action_items=["Finance team to prepare Q2 projections"],
    priority="high"
)

print("Validated object:")
print(valid_memo)
print()
print("As JSON:")
print(valid_memo.model_dump_json(indent=2))

---

## 3. Schema Validation Playground

Pydantic validates the **shape** of data -- types, required fields, structure. But it does NOT validate **meaning**. A string that says `"banana"` in the `author` field is structurally valid even though it is semantically nonsense.

For each example below, **predict** whether it will pass or fail validation **before** running the cell. Then run it to check your answer.

---

### Example 1: Wrong type for `key_points`

`key_points` expects `list[str]`, but we pass a single string.

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_1 = MemoExtraction(
        author="Sarah Chen",
        date="2025-03-15",
        subject="Q1 Budget Review",
        key_points="Just one string, not a list"  # Wrong type!
    )
    print("PASS -- Pydantic accepted it")
    print(bad_1)
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Example 2: Missing a required field

`author` is required, but we leave it out entirely.

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_2 = MemoExtraction(
        # author is missing!
        date="2025-03-15",
        subject="Q1 Budget Review",
        key_points=["Approved marketing spend"]
    )
    print("PASS -- Pydantic accepted it")
    print(bad_2)
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Example 3: Extra field not in the schema

We pass a field called `location` that is not defined in `MemoExtraction`.

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_3 = MemoExtraction(
        author="Sarah Chen",
        date="2025-03-15",
        subject="Q1 Budget Review",
        key_points=["Approved marketing spend"],
        location="Chicago"  # Not in the schema!
    )
    print("PASS -- Pydantic accepted it")
    print(bad_3)
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Example 4: Semantically wrong but structurally valid

`author` is a string. We pass a nonsensical value. Pydantic only checks types, not meaning.

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_4 = MemoExtraction(
        author="BANANA",            # Nonsense, but it IS a string
        date="not-a-real-date",     # Wrong format, but it IS a string
        subject="",                  # Empty, but it IS a string
        key_points=[]               # Empty list, but it IS a list
    )
    print("PASS -- Pydantic accepted it")
    print(bad_4)
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Example 5: Optional field left out

`priority` has `default=None` and `action_items` has `default_factory=list`. What happens when we omit them?

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_5 = MemoExtraction(
        author="Sarah Chen",
        date="2025-01-06",
        subject="Annual Kickoff",
        key_points=["Revenue target: $60M", "AI initiative approved"]
        # action_items and priority are omitted
    )
    print("PASS -- Pydantic accepted it")
    print(f"  action_items = {bad_5.action_items}")
    print(f"  priority     = {bad_5.priority}")
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Example 6: Integer where a string is expected

`date` expects a string, but we pass an integer. Will Pydantic coerce it or reject it?

**Your prediction:** Pass or Fail?

In [ ]:
try:
    bad_6 = MemoExtraction(
        author="Sarah Chen",
        date=20250315,  # Integer, not a string
        subject="Q1 Budget Review",
        key_points=["Approved marketing spend"]
    )
    print("PASS -- Pydantic accepted it")
    print(f"  date = {bad_6.date!r}  (type: {type(bad_6.date).__name__})")
except ValidationError as e:
    print("FAIL -- Pydantic rejected it")
    print(e)

### Playground Takeaway

**Pydantic catches shape problems, not meaning problems.**

- Wrong type? Caught.
- Missing required field? Caught.
- Nonsensical value in a valid type? Passes.
- Empty string or empty list? Passes.
- Extra fields? Ignored by default (Pydantic silently drops them).

This is why extraction quality depends on BOTH the schema AND the prompt. The schema enforces shape. The prompt guides meaning.

---

## 4. Single-Document Extraction

Now we connect the schema to Claude. The `extract_single()` function in `extractor.py`:

1. Converts your Pydantic schema to a JSON Schema string
2. Sends the document text + schema to Claude in a prompt
3. Parses Claude's JSON response
4. Validates the parsed data against the Pydantic model
5. Returns a validated Pydantic object

Let's start with a single memo.

In [ ]:
# Load a document from the Northbrook corpus
doc_path = Path("../data/northbrook/memo_ceo_annual_kickoff.md")
document_text = doc_path.read_text()

# Preview the first 500 characters so you can see what Claude will receive
print(f"Document: {doc_path.name}")
print(f"Length: {len(document_text)} characters")
print("---")
print(document_text[:500])
print("...")

In [ ]:
# Run extraction -- this makes an API call to Claude
result = extract_single(document_text, MemoExtraction)

# The result is a validated Pydantic object
print(f"Type: {type(result).__name__}")
print()
print(result.model_dump_json(indent=2))

In [ ]:
# You can access individual fields as Python attributes
print(f"Author:       {result.author}")
print(f"Date:         {result.date}")
print(f"Subject:      {result.subject}")
print(f"Priority:     {result.priority}")
print(f"Key points:   {len(result.key_points)} items")
for i, point in enumerate(result.key_points, 1):
    print(f"  {i}. {point}")
print(f"Action items: {len(result.action_items)} items")
for i, item in enumerate(result.action_items, 1):
    print(f"  {i}. {item}")

---

## 5. Extraction Deep Dive

Let's look at what is actually happening inside `extract_single()`. Understanding the prompt construction is important because this is where your schema becomes an instruction to Claude.

In [ ]:
# This is the JSON Schema that gets embedded in the prompt
# Claude reads this to understand what fields to extract
schema_json = json.dumps(MemoExtraction.model_json_schema(), indent=2)

print("=== JSON Schema sent to Claude ===")
print(schema_json)

In [ ]:
# The system prompt used by the extractor
from src.s1_extraction.extractor import SYSTEM_PROMPT

print("=== System Prompt ===")
print(SYSTEM_PROMPT)

In [ ]:
# The full user prompt that gets sent -- this is what Claude actually sees
user_prompt = (
    f"Extract data from the following document into this JSON schema:\n\n"
    f"--- SCHEMA ---\n{schema_json}\n\n"
    f"--- DOCUMENT ---\n{document_text}"
)

print("=== User Prompt (first 800 chars) ===")
print(user_prompt[:800])
print("...")
print(f"\n(Total prompt length: {len(user_prompt)} characters)")

**What to notice:**

- The JSON Schema includes every field's `description` -- this is why writing good descriptions matters.
- The system prompt explicitly says "Return ONLY valid JSON" -- this reduces the chance of markdown wrapping.
- The user prompt packages the schema and the document together so Claude has everything it needs in one message.

### Try a Different Document Type

The same `extract_single()` function works with any schema. Let's try extracting meeting notes.

In [ ]:
# Load meeting notes and extract with MeetingExtraction schema
meeting_text = Path("../data/northbrook/engineering_standup_2025_01.md").read_text()

meeting_result = extract_single(meeting_text, MeetingExtraction)

print(f"Meeting date: {meeting_result.meeting_date}")
print(f"Attendees:    {len(meeting_result.attendees)} people")
for name in meeting_result.attendees:
    print(f"  - {name}")
print(f"Decisions:    {len(meeting_result.decisions)} items")
print(f"Action items: {len(meeting_result.action_items)} items")
for item in meeting_result.action_items:
    print(f"  - {item}")
print(f"Next meeting: {meeting_result.next_meeting}")

**Notice:** Same function, different schema, different document type. The schema tells Claude what to look for. The function handles the rest.

---

## 6. Batch Processing

One document is easy. But real pipelines process many documents at once. The `extract_batch()` function:

- Loops over a list of file paths
- Calls `extract_single()` for each document
- Wraps each call in `try/except` so one bad document cannot crash the entire batch
- Returns a list of `ExtractionResult` objects (each knows whether it succeeded or failed)

Let's process a batch of Northbrook memos.

In [ ]:
# List the memo files we will process
data_dir = Path("../data/northbrook")

memo_files = sorted(str(p) for p in data_dir.glob("memo_*.md"))

print(f"Found {len(memo_files)} memo files:")
for f in memo_files:
    print(f"  {f}")

In [ ]:
# Run the batch -- this will make one API call per document
# Watch the output: [OK] for successes, errors for failures
results = extract_batch(memo_files, MemoExtraction)

In [ ]:
# Separate successes and failures
successes = [r for r in results if r.success]
failures = [r for r in results if not r.success]

print(f"\n=== Batch Summary ===")
print(f"Total documents: {len(results)}")
print(f"Successes:       {len(successes)}")
print(f"Failures:        {len(failures)}")

In [ ]:
# Inspect the successful extractions
print("=== Successful Extractions ===\n")
for r in successes:
    print(f"File: {Path(r.file_path).name}")
    print(f"  Author:  {r.data.author}")
    print(f"  Date:    {r.data.date}")
    print(f"  Subject: {r.data.subject}")
    print(f"  Points:  {len(r.data.key_points)} key points")
    print(f"  Actions: {len(r.data.action_items)} action items")
    print()

In [ ]:
# Inspect any failures -- what went wrong and why?
if failures:
    print("=== Failed Extractions ===\n")
    for r in failures:
        print(f"File:  {Path(r.file_path).name}")
        print(f"Error: {r.error[:200]}")
        print()
else:
    print("No failures -- all documents extracted successfully.")

**The key observation:** The batch finished. It did not crash. Even if some documents failed, you know exactly which ones and why. This is the batch processing pattern -- every document gets its own `try/except`.

---

## 7. Results Analysis

Let's dig deeper into what we extracted and look at the data quality.

In [ ]:
# Build a summary table of all successful extractions
print(f"{'File':<35} {'Author':<20} {'Date':<12} {'Points':<8} {'Actions':<8}")
print("-" * 85)
for r in successes:
    name = Path(r.file_path).name
    print(f"{name:<35} {r.data.author:<20} {r.data.date:<12} {len(r.data.key_points):<8} {len(r.data.action_items):<8}")

In [ ]:
# View the full extraction for one document as formatted JSON
if successes:
    print(f"Full extraction for: {Path(successes[0].file_path).name}")
    print(successes[0].data.model_dump_json(indent=2))

In [ ]:
# Data quality check: which fields are populated vs empty?
print("=== Data Quality Check ===\n")
for r in successes:
    name = Path(r.file_path).name
    empty_fields = []
    if not r.data.key_points:
        empty_fields.append("key_points")
    if not r.data.action_items:
        empty_fields.append("action_items")
    if r.data.priority is None:
        empty_fields.append("priority")

    status = "all fields populated" if not empty_fields else f"empty: {', '.join(empty_fields)}"
    print(f"  {name:<35} {status}")

**Observations to discuss:**

- Did Claude find the correct author and date for each memo?
- Are the key points reasonable summaries of the actual content?
- Which optional fields came back empty? Is that correct given the source documents?
- Would you trust this data to go into a database without human review?

---

## 8. Variation Exercises

Pick one of the following. You have 20 minutes.

---

### Option A: Extract a Financial Report

Define a `FinancialSummary` schema below and use `extract_single()` to extract data from the Q3 financial report. Think about what fields a financial summary needs.

### Field Types Quick Reference

Use this table when designing your own schemas:

| Type | Example | When to Use |
|------|---------|-------------|
| `str` | `title: str` | Free-form text |
| `int` | `page_count: int` | Whole numbers |
| `float` | `total_revenue: float` | Money, percentages, measurements |
| `bool` | `confidential: bool` | Yes/no flags |
| `Literal[...]` | `doc_type: Literal["memo", "policy"]` | Fixed set of choices |
| `list[str]` | `departments: list[str]` | Multiple values |
| `str \| None` | `author: str \| None` | Optional fields |

You saw all of these in the `DocumentMeta` schema. Now use them in your own design below.

In [ ]:
# Option A: Define your FinancialSummary schema
#
# Think about: What fields does a financial report have?
# Consider: period, revenue, expenses, net_income, key_metrics, notes
# Remember: Field descriptions guide Claude's extraction!
# TIP: Use float for monetary amounts -- this is a numeric type Pydantic will validate.

class FinancialSummary(BaseModel):
    """Structured extraction for financial reports."""
    period: str = Field(description="The reporting period (e.g., 'Q3 2024')")
    prepared_by: str = Field(description="Who prepared the report")
    total_revenue: float = Field(description="Total revenue for the period as a number")
    total_expenses: float = Field(description="Total operating expenses for the period as a number")
    net_income: float = Field(description="Net income (revenue minus expenses) as a number")
    key_metrics: list[str] = Field(description="Important financial metrics or KPIs mentioned")
    outlook: str | None = Field(default=None, description="Forward-looking statements or projections")

print("FinancialSummary schema defined. Fields:")
for name, field_info in FinancialSummary.model_fields.items():
    print(f"  {name}: {field_info.description}")

In [ ]:
# Option A: Run extraction on the financial report
financial_text = Path("../data/northbrook/financial_report_q3_2024.md").read_text()

financial_result = extract_single(financial_text, FinancialSummary)
print(financial_result.model_dump_json(indent=2))

### Option B: Field Description A/B Test

Open the standalone notebook `notebooks/field_ab_test.ipynb` for a guided experiment comparing weak vs rich field descriptions on the same documents. This empirically proves that descriptions matter.

### Option C: Handle a Tricky Document

Try extracting from a document that does not match the memo schema well. What happens? Can you adjust the extraction to handle it gracefully?

In [ ]:
# Option C: Try extracting a policy document with the MemoExtraction schema
# This is a deliberate mismatch -- what happens?
policy_text = Path("../data/northbrook/remote_work_policy.md").read_text()

try:
    wrong_schema_result = extract_single(policy_text, MemoExtraction)
    print("Extraction succeeded (but is the data correct?)")
    print(wrong_schema_result.model_dump_json(indent=2))
except Exception as e:
    print(f"Extraction failed: {e}")

# Now try with PolicyExtraction -- the correct schema
print("\n--- Now with the correct schema ---\n")
correct_result = extract_single(policy_text, PolicyExtraction)
print(correct_result.model_dump_json(indent=2))

---

## Try It Yourself

Use the cells below to experiment on your own. Some ideas:

- Extract from a different document in `data/northbrook/`
- Try the `MeetingExtraction` schema on the board meeting minutes
- Modify a schema and re-run extraction to see how the output changes
- Extract the same document with two different schemas and compare

In [ ]:
# List all available documents
print("Available documents in data/northbrook/:")
for p in sorted(data_dir.glob("*.md")):
    print(f"  {p.name}")

In [ ]:
# Your experiment here


In [ ]:
# Another experiment here


---

## 9. The Bridge to Embeddings

We now have a way to turn unstructured documents into structured data:

```
Raw memo text  -->  extract_single()  -->  MemoExtraction object  -->  clean JSON
```

But structured data alone is not searchable by meaning. If someone asks *"What did Sarah say about AI?"*, keyword search might miss documents that talk about *"machine learning"* or *"automation"* without using the word *"AI"*.

**Next session (2.1)** we solve this by turning text into **embeddings** -- numerical vectors that capture meaning. Documents with similar meaning end up close together in vector space, even if they use different words.

**Questions to think about before next class:**

- How would you search for documents that are "similar" to a query without using keywords?
- If a faster embedding model is 10% less accurate, when is that acceptable? When is it not?
- What information about a document gets lost when you turn it into numbers?

---

## 10. Session Recap

### What we built today: three pieces that snap together

1. **Schema** = the contract between Claude's output and your code
2. **`extract_single()`** = one document in, one validated object out
3. **`extract_batch()`** = many documents, errors contained

### Key takeaways

- **Pydantic catches shape problems, not meaning problems.** Validation ensures the right types and required fields, but cannot tell you if the extracted content is correct.
- **Field descriptions are prompt engineering.** They guide Claude's extraction. Better descriptions lead to better results.
- **Batch processing must be resilient.** Every document gets its own `try/except`. One bad document must never crash the entire pipeline.
- **The same extraction function works with any schema.** Define a new Pydantic model, and extraction works automatically.

### Connection to Lab 1

The schemas and extraction functions explored today become the core of **Lab 1 (Batch Extraction Pipeline)**, which is assigned in Session 2.2. Start thinking about what document types you want to process and what schemas you would design for them.

### Homework: Extraction Hardening (due before Session 2.1)

1. Create one additional Pydantic schema for a different document type
2. Run `extract_single()` with your new schema against at least 2 documents
3. Save the output as JSON files in `output/`
4. (Stretch) Iterate on the extraction prompt if results are incorrect